# ONCOVA 데모: 변이 목록 → 암 종류 후보 Top-3

- 목적: NGS 검사로 찾은 **유전자 변이 목록**을 넣으면, 암이 처음 생긴 장기 후보 3개와 확률, 근거 유전자를 보여주는 웹 화면을 만듦
- 사용 모델: `02_v2_hotspot.ipynb`에서 저장한 v2-N5 (XGBoost + 변이 유전자 수 + 핫스팟 573개, 11/9 리더보드 0.3236)
- 화면 도구: Gradio (실행하면 다른 사람에게 보낼 수 있는 임시 공유 링크가 나옴)
- 위치: 연구용 프로토타입이며 진단이 아님. 최종 판단은 의료진이 함 ([6_1 규제·윤리](./docs/6_1_규제윤리_박종부.md))

**데이터 사용 원칙**
- 이 노트북은 대회 데이터(train/test)를 불러오지 않음. 필요한 것은 모델 파일 하나에 모두 들어 있음
- 예시 환자는 잘 알려진 변이로 만든 **가상 환자**임. 대회 데이터의 환자는 공유 링크로 외부에 보여주면 안 되므로 쓰지 않음 (대회 규칙 제7조)

## 0. 설치 (처음 한 번만)

- 웹 화면을 만드는 `gradio`를 설치함
- 설치가 끝나면 **Kernel → Restart** 를 한 번 해야 다음 셀에서 불러올 수 있음

In [ ]:
%pip install gradio

## 1. 모델 불러오기

- `output/model_v2_hotspot.pkl` 하나에 학습된 모델, 라벨 인코더(숫자 ↔ 아형 이름), 유전자 4,384개 순서, 핫스팟 573개 목록이 모두 들어 있음
- `유전자 4384 / 핫스팟 573 / 아형 26`이 나오면 정상임
- 02 노트북을 다시 돌려서 모델이 바뀌면 이 파일도 바뀜 → 데모도 새 모델로 바뀜

In [ ]:
import re                      # 변이 표기에서 숫자(위치)를 찾는 도구
import joblib                  # 저장한 모델 파일을 불러옴
import numpy as np             # 수치 계산
import pandas as pd            # 표 데이터 처리
import xgboost as xgb          # 근거 유전자(기여도) 계산에 사용
import gradio as gr            # 웹 데모 화면

# 02 노트북에서 저장한 모델 묶음 (모델, 라벨 인코더, 유전자 목록, 핫스팟 목록)
art = joblib.load('./output/model_v2_hotspot.pkl')
model = art['model']
le = art['label_encoder']
gene_cols = art['gene_cols']
hotspots = art['hotspots']
print('유전자', len(gene_cols), '/ 핫스팟', len(hotspots), '/ 아형', len(le.classes_))

## 2. 피처 만드는 함수 (02 노트북에서 가져옴)

- 예측할 때도 **학습 때와 똑같은 방식**으로 피처를 만들어야 해서, `02_v2_hotspot.ipynb`의 함수를 그대로 복사함
- `make_features`: 유전자별 변이 있음 1 / 없음 0 + 변이 유전자 수(`n_mut`)
- `get_positions`: 변이 표기에서 위치 숫자를 뽑음 (`'E726K H1047R'` → `['726', '1047']`)
- `make_hotspot_features`: 핫스팟(예: `BRAF_600`)마다 그 자리에 변이가 있으면 1
- 함수 정의만 가져오고, 아래에 있던 `make_features(train, ...)` 같은 실행 줄은 뺌 (이 노트북에는 `train`이 없음)

In [ ]:
def make_features(df, gene_cols):
    # train에서 정한 유전자 순서대로 컬럼을 맞춤 (혹시 없으면 WT로 채움)
    genes = df.reindex(columns=gene_cols, fill_value='WT')
    # WT가 아니면 1, WT면 0
    X = pd.DataFrame((genes.to_numpy() != 'WT').astype(np.float32), columns=gene_cols, index=df.index)
    # 환자 한 명이 가진 변이 유전자 수
    X['n_mut'] = X.sum(axis=1)
    return X

In [ ]:
def get_positions(cell):
    # cell 예: 'V600E' 또는 'Q369* I368N' 또는 'WT'
    if cell == 'WT':                    # 변이가 없으면
        return []                       # 빈 리스트를 돌려줌
    positions = []                      # 위치 숫자를 모을 리스트
    for mut in cell.split():            # 공백 기준으로 나눔
        nums = re.findall(r'\d+', mut)  # 지금 보고 있는 변이 하나
        if nums:                        # 숫자가 하나라도 있으면
            positions.append(nums[0])   # 첫 번째 숫자 위치
    return positions

In [ ]:
def make_hotspot_features(df, hotspots):
    cols = {}                                      # 핫스팟 이름 → 0/1 값을 모을 딕셔너리
    for hs in hotspots:                            # 예: 'BRAF_600'
        gene, pos = hs.rsplit('_', 1)              # 'BRAF', '600'으로 나눔 (오른쪽 '_' 한 번만)
        # 그 유전자 칸에 해당 위치가 있으면 1, 없으면 0
        cols[hs] = df[gene].apply(lambda c: pos in get_positions(c)).astype(np.float32)
    return pd.DataFrame(cols, index=df.index)      # 모은 열을 한 번에 표로 만듦

print(get_positions('E726K H1047R'))   # 확인용

## 3. 입력 글자 → 유전자별 변이

- 사용자가 `TP53 R175H, BRAF V600E`처럼 입력하면 `{'TP53': 'R175H', 'BRAF': 'V600E'}`로 바꿈
- 쉼표로 나누고, 첫 번째 띄어쓰기에서 유전자 이름과 변이 표기를 나눔
- 유전자 이름은 대문자로 통일함 (`braf` → `BRAF`)
- 같은 유전자가 두 번 나오면 대회 데이터처럼 공백으로 이어 붙임 (`PIK3CA E726K H1047R`)

In [ ]:
def parse_input(text):
    muts = {}                                   # 유전자 → 변이 표기
    for item in text.split(","):                # 쉼표로 나눔
        item = item.strip()                     # 앞뒤 공백 제거
        if not item:                            # 빈 칸이면 건너뜀
            continue
        gene, variant = item.split(' ', 1)      # 'TP53 R175H' → 'TP53', 'R175H' (첫 공백에서 한 번만 나눔)
        gene = gene.upper()                     # 유전자 이름을 대문자로 통일 (tp53 → TP53)
        # 같은 유전자가 두 번 나오면 데이터처럼 공백으로 이어 붙임 (예: 'E726K H1047R')
        muts[gene] = (muts[gene] + ' ' + variant) if gene in muts else variant
    return muts

print(parse_input('TP53 R175H, braf V600E, PIK3CA E726K, PIK3CA H1047R'))

## 4. 예측 (Top-3)

- `build_features`: 환자 한 명의 표를 만들고 학습 때 유전자 순서로 맞춘 뒤, 0/1 + `n_mut` + 핫스팟 피처를 붙임. 입력하지 않은 유전자는 WT(변이 없음)로 채움
- `predict_top3`: 26개 아형의 확률 중 상위 3개를 % 로 돌려줌
- 학습 데이터에 없는 유전자(예: KRAS)는 모델이 쓸 수 없으므로 따로 모아서 화면에 알려줌
- 예: `BRAF V600E` 하나만 넣으면 THCA(갑상선암) 94.6%

In [ ]:
def build_features(muts):
    # 환자 한 명 표를 만들고, 학습 때 유전자 순서로 맞춤 (입력 안 한 유전자는 WT)
    row = pd.DataFrame([muts]).reindex(columns=gene_cols, fill_value='WT').fillna('WT')
    X = make_features(row, gene_cols)                                     # 0/1 + n_mut
    return pd.concat([X, make_hotspot_features(row, hotspots)], axis=1)   # + 핫스팟

def predict_top3(text):
    muts = parse_input(text)
    X = build_features(muts)
    proba = model.predict_proba(X)[0]                   # 26개 아형별 확률
    top = np.argsort(proba)[::-1][:3]                   # Top 3
    result = [(le.classes_[i], round(float(proba[i]) * 100, 1)) for i in top]   # 확률을 %로 바꿈
    unknown = [g for g in muts if g not in gene_cols]   # 데이터에 없는 유전자 (예: KRAS)는 어떤 목록에 없을까?
    return result, unknown, X, top

print(predict_top3('BRAF V600E')[:2])

## 5. 근거 유전자

- XGBoost의 `pred_contribs` 기능으로 "이 환자를 1등 아형으로 고를 때 각 피처가 얼마나 밀어줬는지(기여도)"를 계산함
- 그중 **환자가 실제로 가진 변이**이면서 1등 아형 쪽으로 밀어준 것만, 큰 순서로 3개 뽑음
- `n_mut`(변이 개수)은 유전자가 아니라서 근거 목록에서는 뺌
- 주의: 근거 유전자는 "모델이 판단에 많이 참고한 변이"이지, 그 암의 **원인 유전자라는 뜻은 아님** ([3_3 모델 해석](./docs/3_3_모델_중요피처_혼동행렬.md))
- 예: `IDH1 R132H, TP53 R273H, ATRX R1426*` → LGG 66.4%, 근거 `IDH1_132, IDH1, ATRX`

In [ ]:
def evidence_genes(X, top_class, k=3):
    booster = model.get_booster()
    contrib = booster.predict(xgb.DMatrix(X), pred_contribs=True)[0]   # (아형 26개, 피처 수 + 1)
    c = contrib[top_class, :-1]                    # 1등 아형에 대한 피처별 기여도 (마지막 칸은 기본값이라 뺌)
    names = np.array(X.columns)
    present = X.iloc[0].to_numpy() > 0             # 이 환자가 실제로 가진 변이만
    order = [i for i in np.argsort(-c)             # 기여도가 큰 순서로
             if present[i] and c[i] > 0 and names[i] != 'n_mut'][:k]   # 가진 변이 + 밀어준 것만, 변이 개수는 제외
    return [names[i] for i in order]

result, unknown, X, top = predict_top3('IDH1 R132H, TP53 R273H, ATRX R1426*')
print(result, evidence_genes(X, top[0]))

## 6. 한글 암 이름과 가상 예시 환자

- `KOR`: 26개 아형 약어를 한글 이름으로 바꿔 보여줌 (새봄님 [1_1 문서](./docs/1_1_NGS_암아형_배경_김새봄.md) 표 기준). 일반 사용자도 결과를 읽을 수 있게 함
- `CASES`: 각 암에서 잘 알려진 변이로 만든 **가상 환자 8명**. 용어를 몰라도 고르기만 하면 써볼 수 있게 함
  - 예시 1~7은 모델이 떠올린 암을 맞힘
  - 예시 8 흑색종은 일부러 남긴 **어려운 예시**임. 흑색종은 실제로 변이가 수십~수백 개인 암이라, 4개만 넣으면 같은 BRAF V600E가 흔한 갑상선암으로 헷갈림 → 모델의 한계를 솔직하게 보여줌

In [ ]:
# 26개 아형 한글 이름 (1_1 문서 표 기준) → 일반 사용자도 읽을 수 있게 함
KOR = {
    'ACC': '부신피질암', 'BLCA': '방광암', 'BRCA': '유방암', 'CESC': '자궁경부암',
    'COAD': '대장암', 'DLBC': '림프종', 'GBMLGG': '뇌종양(교모세포종·저등급 신경교종 묶음)',
    'HNSC': '두경부암', 'KIPAN': '신장암 묶음', 'KIRC': '신장 투명세포암', 'LAML': '급성 골수성 백혈병',
    'LGG': '저등급 신경교종(뇌종양)', 'LIHC': '간암', 'LUAD': '폐선암', 'LUSC': '폐 편평세포암',
    'OV': '난소암', 'PAAD': '췌장암', 'PCPG': '갈색세포종·부신경절종', 'PRAD': '전립선암',
    'SARC': '육종', 'SKCM': '피부 흑색종', 'STES': '위·식도암', 'TGCT': '고환암',
    'THCA': '갑상선암', 'THYM': '흉선종', 'UCEC': '자궁내막암',
}

# 가상 예시 환자 (실제 환자 아님): 각 암에서 잘 알려진 변이로 만든 예시
# ※ 대회 데이터(train/test)의 환자는 외부에 공유하면 안 되므로 쓰지 않음 (대회 규칙 제7조)
CASES = {
    '예시 1 · 갑상선암': ('BRAF V600E', 'THCA'),
    '예시 2 · 대장암': ('APC R1450*, TP53 R175H, PIK3CA E545K, FBXW7 R465H', 'COAD'),
    '예시 3 · 뇌종양(저등급 신경교종)': ('IDH1 R132H, TP53 R273H, ATRX R1426*', 'LGG'),
    '예시 4 · 신장암': ('VHL W117C', 'KIRC'),
    '예시 5 · 유방암': ('PIK3CA H1047R, CDH1 Q23*, GATA3 D336fs, MAP3K1 E1286*', 'BRCA'),
    '예시 6 · 급성 골수성 백혈병': ('NPM1 W288fs, IDH2 R140Q', 'LAML'),
    '예시 7 · 자궁내막암': ('PTEN R130G, PIK3CA H1047R, CTNNB1 S37F', 'UCEC'),
    '예시 8 · 흑색종 (어려운 예시)': ('BRAF V600E, CDKN2A R80*, PTEN R233*, TP53 P278S', 'SKCM'),
}
print(len(KOR), '개 아형 /', len(CASES), '개 예시')

## 7. 데모 화면 실행

**화면 구성**
- 왼쪽: 1. 예시 환자 선택 → 2. 변이 목록(직접 입력 및 수정 가능) → [🔄 새로고침] [예측하기] (어두운 회색 계열) → 변이 표기 읽는 법
- 오른쪽: 암 종류 후보 3개 표(순위, 한글 이름, 확률), 예시를 만들 때 떠올린 암과 일치 여부, 근거 유전자, 주의 문구
- 맨 위: 제목(아래쪽 테두리) + 오른쪽 위 **☀ Light / ☾ Dark** 알약 버튼(테마 전환)
- 꾸미기는 `CSS`에서 함: 좌우 칸 간격(`.main-row`), 새로고침 버튼 모양(`.reset-btn`), 제목 테두리(`.title-row`), 예측하기(`.predict-btn`), 테마 버튼(`.theme-btn`)
- 휴대폰(폭 640px 이하)에서는 `@media` 규칙으로 테마 버튼을 제목 위로 올리고 제목 글자를 줄임 (좁은 칸에 제목이 한 글자씩 꺾이던 문제 해결)
- `gr.close_all()`이 이전 서버를 닫아서, 이 셀만 다시 실행해도 서버가 하나만 유지됨

**규제·윤리 반영** ([6_1](./docs/6_1_규제윤리_박종부.md))
- "진단" 대신 "예측 보조", "연구용 프로토타입(Research Use Only)" 표시
- 1등 확률이 40% 미만이면 **판단 보류** 를 권함 (`HOLD`)
- 근거 유전자 옆에 "원인 유전자라는 뜻은 아님" 주석

**실행 결과**
- `Running on public URL: https://….gradio.live` 주소를 보내면 다른 사람도 써볼 수 있음 (이 노트북이 켜져 있는 동안만 작동)
- 피드백 질문지: [5_5 데모 피드백 질문지](./docs/5_5_데모_피드백_질문지.md)

In [ ]:
HOLD = 40   # 1등 확률이 이보다 낮으면 "판단 보류" 표시 (%)

HOW_TO = """
**💡 변이 표기 읽는 법**  
`BRAF V600E` = BRAF 유전자의 **600번째** 아미노산이 **V**(발린)에서 **E**(글루탐산)로 바뀜  
`R1450*` = 1450번째에서 단백질이 **끊김** / `W288fs` = 288번째부터 읽는 틀이 **밀림**  
여러 개는 쉼표로 구분합니다. 예: `BRAF V600E, TP53 R175H`
"""

def name_kor(code):
    return f'{code} {KOR.get(code, "")}'          # 'THCA' → 'THCA 갑상선암'

def run(text, case_name):
    if not text.strip():
        return '왼쪽에서 **예시 환자를 고르거나** 변이를 직접 입력해주세요.'
    try:
        result, unknown, X, top = predict_top3(text)
    except ValueError:                             # 'BRAFV600E'처럼 띄어쓰기가 없으면 나눌 수 없음
        return '⚠️ 입력 형식을 확인해주세요. 유전자와 변이 사이에 띄어쓰기가 필요합니다. 예: `BRAF V600E`'
    genes = evidence_genes(X, top[0])
    # 후보 3개를 표로 보여줌
    table = ['| 순위 | 암 종류 | 확률 |', '|:---:|---|---:|']
    for rank, (code, pct) in enumerate(result, 1):
        table.append(f'| {rank} | **{code}** {KOR.get(code, "")} | {pct}% |')
    lines = ['### 암 종류 후보 (예측 보조)', '\n'.join(table)]
    if result[0][1] < HOLD:
        lines.append('⚠️ 확신도가 낮아 **판단 보류** 를 권합니다.')
    # 가상 예시를 그대로 넣은 경우: 만들 때 떠올린 암과 비교해서 보여줌
    if case_name in CASES and CASES[case_name][0] == text.strip():
        expect = CASES[case_name][1]
        mark = '✅ 일치' if result[0][0] == expect else '❌ 다름'
        lines.append(f'**예시를 만들 때 떠올린 암**: {name_kor(expect)} → {mark}')
        if expect == 'SKCM':
            lines.append('※ 흑색종은 실제로 변이가 매우 많은 암이라(환자 1명당 수십~수백 개), 이렇게 몇 개만 넣으면 같은 BRAF V600E가 흔한 갑상선암으로 헷갈릴 수 있습니다.')
    lines.append(f'**근거 유전자**: {", ".join(genes) if genes else "없음"}')
    lines.append('※ 모델이 판단에 많이 참고한 변이이며, 암의 원인 유전자라는 뜻은 아닙니다.')
    if unknown:
        lines.append(f'※ 학습 데이터에 없는 유전자라 반영하지 않음: {", ".join(unknown)}')
    lines.append('※ 연구용 프로토타입(Research Use Only)입니다. 진단이 아니며, 최종 판단은 의료진이 합니다.')
    return '\n\n'.join(lines)                     # 빈 줄로 이어야 화면에서 줄이 나뉨

def fill_case(case_name):
    return CASES[case_name][0] if case_name in CASES else ''   # 예시를 고르면 입력칸을 채움

# 화면 꾸미기 (CSS)
CSS = """
/* 좌우 칸 간격 */
.main-row { gap: 48px !important; }

/* 예측하기: 어두운 회색 / 새로고침: 같은 계열 테두리 버튼 */
.predict-btn { background: #374151 !important; color: white !important; border: 1px solid #374151 !important; }
.predict-btn:hover { background: #1f2937 !important; }
.reset-btn { background: white !important; color: #374151 !important; border: 1px solid #9ca3af !important; }
.reset-btn:hover { background: #f3f4f6 !important; }
.dark .predict-btn { background: #4b5563 !important; border-color: #4b5563 !important; }
.dark .predict-btn:hover { background: #6b7280 !important; }
.dark .reset-btn { background: transparent !important; color: #e5e7eb !important; border-color: #6b7280 !important; }
.dark .reset-btn:hover { background: rgba(255, 255, 255, 0.06) !important; }

/* 제목 줄: 아래쪽 테두리 */
.title-row { border-bottom: 1px solid var(--border-color-primary); padding-bottom: 12px; margin-bottom: 8px; align-items: center; }

/* 오른쪽 위 Light / Dark 알약 버튼 (감싸는 테두리 없음) */
.theme-row { gap: 8px !important; flex-wrap: nowrap !important; justify-content: flex-end; background: none !important; border: none !important; }
.theme-btn { border-radius: 999px !important; border: 1px solid #e5e7eb !important; background: white !important;
             color: #9ca3af !important; font-weight: 500 !important; box-shadow: none !important; min-width: 92px !important; }
.theme-btn.active { color: #374151 !important; border-color: #9ca3af !important; }
.dark .theme-btn { background: transparent !important; border-color: #4b5563 !important; color: #6b7280 !important; }
.dark .theme-btn.active { color: #e5e7eb !important; border-color: #9ca3af !important; }

/* 휴대폰 화면 (폭 640px 이하): 제목이 좁게 눌리지 않도록 테마 버튼을 제목 위로 올리고 글자를 줄임 */
@media (max-width: 640px) {
    .title-row { flex-direction: column-reverse !important; align-items: stretch !important; gap: 4px !important; }
    .title-row > * { min-width: 100% !important; }
    .title-row h1 { font-size: 1.5rem !important; line-height: 1.3 !important; word-break: keep-all; }   /* 단어 중간에서 줄이 꺾이지 않게 */
    .theme-row { justify-content: flex-end !important; }
    .theme-btn { min-width: 72px !important; flex: 0 0 auto !important; }   /* 버튼이 화면 폭만큼 늘어나지 않게 */
    .main-row { gap: 24px !important; }
}
"""

# Light / Dark 전환: Gradio는 화면 맨 바깥(body)에 'dark' 클래스가 있으면 다크 테마로 그림
def theme_js(mode):
    return f"""
    () => {{
        if ('{mode}' === 'dark') {{ document.body.classList.add('dark'); }}
        else {{ document.body.classList.remove('dark'); }}
        document.querySelectorAll('.theme-btn').forEach(b => b.classList.remove('active'));
        document.querySelector('#{mode}-btn').classList.add('active');   // 지금 테마 버튼만 진하게
    }}
    """

gr.close_all()                                    # 이전에 띄운 데모 서버가 있으면 닫음 (여러 번 실행해도 하나만 유지)

with gr.Blocks(title='ONCOVA 데모') as demo:
    with gr.Row(elem_classes='title-row'):                  # 제목 줄 (아래쪽 테두리) + 오른쪽 테마 전환
        gr.Markdown('# ONCOVA 암 종류 예측 보조 (연구용 프로토타입)\n'
                    'NGS 검사로 찾은 **유전자 변이 목록**을 넣으면, 암이 처음 생긴 장기 후보 3개와 근거 유전자를 보여줍니다.')
        with gr.Row(elem_classes='theme-row', scale=0):        # 감싸는 테두리 없이 버튼만 나란히
            light_btn = gr.Button('☀ Light', size='sm', elem_id='light-btn', elem_classes='theme-btn')
            dark_btn = gr.Button('☾ Dark', size='sm', elem_id='dark-btn', elem_classes='theme-btn')
    with gr.Row(elem_classes='main-row'):                   # 왼쪽 입력 / 오른쪽 결과
        with gr.Column():
            case = gr.Dropdown(list(CASES), value=None, label='1. 예시 환자 선택 (가상 환자)')   # 처음엔 아무것도 안 고른 상태
            text = gr.Textbox(label='2. 변이 목록 (직접 입력 및 수정 가능)', lines=3,
                              placeholder='예: BRAF V600E, TP53 R175H')
            with gr.Row():                                  # 버튼 두 개를 나란히
                reset = gr.Button('🔄 새로고침', elem_classes='reset-btn')
                btn = gr.Button('예측하기', elem_classes='predict-btn')
            gr.Markdown(HOW_TO)
        with gr.Column():
            out = gr.Markdown()
    case.change(fill_case, inputs=case, outputs=text)                 # 예시를 고르면 입력칸 채우기
    btn.click(run, inputs=[text, case], outputs=out)                 # 예측하기
    reset.click(lambda: (None, '', ''), outputs=[case, text, out])   # 예시·입력·결과를 모두 비움
    light_btn.click(None, js=theme_js('light'))                      # Light / Dark 전환 (브라우저에서 바로 바뀜)
    dark_btn.click(None, js=theme_js('dark'))
    demo.load(None, js=theme_js('light'))                             # 처음에는 Light로 시작

demo.launch(share=True, css=CSS)
